|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>CUDA graphs<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: capture a step, then survive a changing batch<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import cudalib

Capture a decode step as a CUDA graph, then make it survive a batch size
that changes every step.

This is stage 12. The capture is twenty lines and three of them are traps.

In [ ]:
### run this cell: a stand-in decode step

class Block(nn.Module):
  def __init__(self, d):
    super().__init__()
    self.n1, self.n2 = nn.RMSNorm(d), nn.RMSNorm(d)
    self.qkv = nn.Linear(d, 3*d, bias=False)
    self.o   = nn.Linear(d, d, bias=False)
    self.up  = nn.Linear(d, 4*d, bias=False)
    self.dn  = nn.Linear(4*d, d, bias=False)
  def forward(self, x):
    h = self.n1(x); q,k,v = self.qkv(h).chunk(3,-1)
    x = x + self.o(torch.softmax(q @ k.transpose(-1,-2)/32.0, -1) @ v)
    h = self.n2(x)
    return x + self.dn(F.silu(self.up(h)))

class Model(nn.Module):
  def __init__(self, n=28, d=1024):
    super().__init__(); self.blocks = nn.ModuleList([Block(d) for _ in range(n)])
  def forward(self, x):
    for b in self.blocks: x = b(x)
    return x

D = 1024
model = Model(28, D).cuda().to(torch.bfloat16).eval()
model.requires_grad_(False)
print('28 layers, hidden', D)

# Exercise 1: capture and replay

Warm up, record, replay. Measure what it bought.

In [ ]:
def capture(model, example):
  """Record one forward pass. Returns (graph, static_input, static_output)."""
  static_in = example.clone()

  # warm up on a SIDE STREAM first. cuBLAS allocates workspaces on first
  # use and you must not record that allocation into the graph.
  s = torch.cuda.Stream(); s.wait_stream(torch.cuda.current_stream())
  with torch.cuda.stream(s):
    
  torch.cuda.current_stream().wait_stream(s)

  g = torch.cuda.CUDAGraph()
  with torch.cuda.graph(g):
    static_out = 
  return g, static_in, static_out

with torch.no_grad():
  x = torch.randn(1, 1, D, device='cuda', dtype=torch.bfloat16)
  g, si, so = capture(model, x)
  eager = 
graph = 
print(f'eager {eager:.3f} ms, graph {graph:.3f} ms  ({eager/graph:.2f}x)')

# Exercise 2: feed it

Replay runs the exact work that the capture recorded. It reads and writes the
exact buffers that the capture recorded. Put your data into those buffers.

In [ ]:
real = torch.randn(1, 1, D, device='cuda', dtype=torch.bfloat16)

# get `real` into the graph's input. Careful: the graph holds a POINTER.

g.replay()
from_graph = so.clone()

with torch.no_grad():
  from_eager = model(real)
print('agree:', torch.allclose(from_graph, from_eager, rtol=1e-2, atol=1e-2))

# now do it the wrong way and see what happens
si = torch.randn(1, 1, D, device='cuda', dtype=torch.bfloat16)   # rebinding!
g.replay()
print('still the OLD output:', torch.allclose(so, from_graph))

# Exercise 3: shape buckets

A server's batch size changes every step and a graph's does not. Capture
several and pad up to the nearest.

In [ ]:
BUCKETS = [1, 2, 4, 8, 16, 32]
graphs = {}
with torch.no_grad():
  for b in BUCKETS:
    graphs[b] = 

def bucket_for(n):
  # the smallest bucket that fits n, or None if n is bigger than all of them
  return 

def run(n):
  b = bucket_for(n)
  if b is None:
    with torch.no_grad(): return model(torch.randn(n,1,D,device='cuda',dtype=torch.bfloat16))
  gg, sin, sout = graphs[b]
  # copy n rows into the first n rows of the bucket's input, replay,
  # and read back only the n rows you care about
  
  return 

print(f"{'seqs':>5} {'bucket':>7} {'padding':>8} {'eager ms':>10} {'graph ms':>10} {'gain':>6}")
for n in (1, 3, 5, 12, 31, 48):
  xn = torch.randn(n,1,D,device='cuda',dtype=torch.bfloat16)
  with torch.no_grad():
    e = cudalib.bench_ms(lambda: model(xn), iters=30, warmup=10, best_of=2)
  gm = cudalib.bench_ms(lambda: run(n), iters=30, warmup=10, best_of=2)
  b = bucket_for(n)
  pad = f'{100*(b-n)/b:.0f}%' if b else 'n/a'
  print(f'{n:>5} {str(b):>7} {pad:>8} {e:>10.3f} {gm:>10.3f} {e/gm:>5.2f}x')

# Exercise 4: what the buckets cost

Not time. Memory.

In [ ]:
# every captured graph holds its own buffers for the life of the server.
# measure what two more buckets cost.
free_before = torch.cuda.mem_get_info()[0]
extra = {}
with torch.no_grad():
  for b in (64, 128):
    
free_after = torch.cuda.mem_get_info()[0]

cost = 
print(f'2 more graphs cost {cost:.0f} MB of VRAM')
print(f'at 112 KB per token of KV cache, that is '
      f'{cost*1e6/(112*1024):,.0f} tokens the pool does not get')

### Before you open the solution

1. Remove the warm-up on the side stream in `capture`, and run it again. What
   changes? Does it fail loudly or quietly?
2. The second half of Exercise 2 binds `si` to a new tensor and replays. You
   get the old answer, and no error. What exactly does the graph hold?
3. Exercise 4 measured the memory that two more buckets cost. Where does that
   memory come from? What did Part 3 do to the same pool in four sections?
4. A step with 5 sequences runs the bucket for 8. The GPU does three rows of
   arithmetic that nobody uses. Why does that cost almost nothing here? Which
   plot from Part 1 tells you?